In [ ]:
import os, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.signal import butter, sosfilt
import pyarrow.parquet as pq
import timm
from tqdm.notebook import tqdm

COMP_DIR = '/kaggle/input/hms-harmful-brain-activity-classification'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

In [ ]:
# ── Find checkpoints ─────────────────────────────────────────────────────────
# Шукаємо всі .ckpt файли в /kaggle/input/ з 'coolz' в шляху.
# Кожен файл = найкращий чекпоінт для одного фолда → ансамблюємо всі.

def find_coolz_ckpts():
    ckpts = []
    for root, _, files in os.walk('/kaggle/input'):
        for f in files:
            if f.endswith('.ckpt'):
                full = os.path.join(root, f)
                if 'coolz' in full.lower() or 'coolz' in f.lower():
                    ckpts.append(full)
    return sorted(ckpts)

print('=== Всі .ckpt у /kaggle/input ===')
found_any = False
for root, _, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.ckpt'):
            print(' ', os.path.join(root, f))
            found_any = True
if not found_any:
    print('  (не знайдено — додай датасет з вагами як Input)')

CKPTS = find_coolz_ckpts()
print(f'\ncoolz checkpoints: {len(CKPTS)}')
for c in CKPTS:
    print(' ', c)

In [ ]:
# ── Preprocessing (має збігатись з coolz/dataset.py) ─────────────────────────

FS = 200
WIN_SAMPLES = 10_000
VOTE_COLS = ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']

DOUBLE_BANANA = [
    ('Fp1', 'F7'), ('F7', 'T3'), ('T3', 'T5'), ('T5', 'O1'),
    ('Fp2', 'F8'), ('F8', 'T4'), ('T4', 'T6'), ('T6', 'O2'),
    ('Fp1', 'F3'), ('F3', 'C3'), ('C3', 'P3'), ('P3', 'O1'),
    ('Fp2', 'F4'), ('F4', 'C4'), ('C4', 'P4'), ('P4', 'O2'),
]
EEG_COLS = list(dict.fromkeys(c for pair in DOUBLE_BANANA for c in pair))
LR_FLIP = [4, 5, 6, 7, 0, 1, 2, 3, 12, 13, 14, 15, 8, 9, 10, 11]

_SOS = butter(5, [0.5, 20.0], btype='bandpass', fs=FS, output='sos')


def load_signal(eeg_id: int, offset_sec: float, eeg_dir: str) -> np.ndarray:
    raw = pq.read_table(f'{eeg_dir}/{eeg_id}.parquet', columns=EEG_COLS).to_pandas()
    start = int(offset_sec * FS)
    chunk = raw.iloc[start: start + WIN_SAMPLES]
    sig = np.stack(
        [chunk[a].values - chunk[b].values for a, b in DOUBLE_BANANA], axis=0
    ).astype(np.float32)
    if sig.shape[1] < WIN_SAMPLES:
        sig = np.pad(sig, ((0, 0), (0, WIN_SAMPLES - sig.shape[1])))
    sig = np.nan_to_num(sig, nan=0.0, posinf=1024.0, neginf=-1024.0)
    sig = np.clip(sig, -1024.0, 1024.0) / 32.0
    return sosfilt(_SOS, sig, axis=-1).astype(np.float32)

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────

class CoolzTestDataset(Dataset):
    def __init__(self, df, eeg_dir):
        self.df = df.reset_index(drop=True)
        self.eeg_dir = eeg_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sig = load_signal(
            int(row['eeg_id']),
            float(row.get('eeg_label_offset_seconds', 0)),
            self.eeg_dir,
        )
        return torch.from_numpy(sig), torch.from_numpy(sig[LR_FLIP])

In [ ]:
# ── Model (має збігатись з coolz/model.py) ───────────────────────────────────

class EEGNet(nn.Module):
    def __init__(self, backbone='efficientnet_b5', dropout=0.5):
        super().__init__()
        self.net = timm.create_model(
            backbone, pretrained=False, in_chans=3,
            num_classes=0, global_pool='',
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(self.net.num_features, 6)

    def forward(self, x):
        B = x.size(0)
        x = x.view(B, 16, 1000, 10).permute(0, 1, 3, 2).reshape(B, 160, 1000)
        x = x.unsqueeze(1).expand(-1, 3, -1, -1).contiguous()
        x = self.net(x)
        x = self.pool(x).view(B, -1)
        x = self.drop(x)
        return F.log_softmax(self.fc(x), dim=1)


def load_model(ckpt_path):
    """
    Завантажує модель з чекпоінту.
    Новий формат: {'backbone': ..., 'dropout': ..., 'state_dict': ...}
    Старий формат: bare state_dict (потрібно вручну встановити FALLBACK_BACKBONE).
    """
    raw = torch.load(ckpt_path, map_location=device, weights_only=True)

    if isinstance(raw, dict) and 'state_dict' in raw:
        backbone = raw.get('backbone', FALLBACK_BACKBONE)
        dropout = raw.get('dropout', 0.5)
        state = raw['state_dict']
    else:
        # старий формат — bare state_dict
        backbone = FALLBACK_BACKBONE
        dropout = 0.5
        state = raw

    model = EEGNet(backbone, dropout).to(device)
    model.load_state_dict(state)
    model.eval()
    print(f'  loaded [{backbone}]: {os.path.basename(ckpt_path)}')
    return model

In [ ]:
# ── НАЛАШТУВАННЯ ─────────────────────────────────────────────────────────────
# Для нових чекпоінтів (збережених coolz/train.py після фіксу) backbone
# читається з файлу автоматично.
#
# Для старих чекпоінтів (bare state_dict) — встанови FALLBACK_BACKBONE вручну.
# Як визначити: якщо у помилці розмір block1=32 → b4, block1=40 → b5.

FALLBACK_BACKBONE = 'efficientnet_b4'  # ← змінити якщо потрібно
BATCH_SIZE = 8

# ── Test data ─────────────────────────────────────────────────────────────────
test_df = pd.read_csv(os.path.join(COMP_DIR, 'test.csv'))
sub = pd.read_csv(os.path.join(COMP_DIR, 'sample_submission.csv'))

if 'eeg_label_offset_seconds' not in test_df.columns:
    test_df['eeg_label_offset_seconds'] = 0.0

eeg_dir = os.path.join(COMP_DIR, 'test_eegs')
ds = CoolzTestDataset(test_df, eeg_dir)
loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'test samples : {len(test_df)}')
print(f'checkpoints  : {len(CKPTS)}')
if not CKPTS:
    raise RuntimeError('Не знайдено coolz чекпоінтів. Додай датасет з вагами.')

In [ ]:
# ── Inference: ensemble + TTA (LR flip) ──────────────────────────────────────

all_fold_probs = []

for ckpt_path in CKPTS:
    print(f'\n→ {os.path.basename(ckpt_path)}')
    model = load_model(ckpt_path)

    probs_orig, probs_flip = [], []
    with torch.no_grad():
        for sig, sig_flip in tqdm(loader, desc='inference', leave=False):
            probs_orig.append(model(sig.to(device)).exp().cpu().numpy())
            probs_flip.append(model(sig_flip.to(device)).exp().cpu().numpy())

    p = (np.concatenate(probs_orig) + np.concatenate(probs_flip)) / 2
    all_fold_probs.append(p)
    print(f'  mean pred: {p.mean(axis=0).round(3)}')

final_probs = np.mean(all_fold_probs, axis=0)
print(f'\nEnsemble: {len(all_fold_probs)} checkpoint(s)')
print(f'final mean: {final_probs.mean(axis=0).round(3)}')

In [ ]:
# ── Submission ────────────────────────────────────────────────────────────────

sub[VOTE_COLS] = final_probs
sub.to_csv('submission.csv', index=False)

print('Saved: submission.csv')
print(sub.head())